In [ ]:
#| default_exp network

# Network

> VPC / Firewall rules, Secret Manager, Service Accounts, Private Service Connect,
> Cloud CDN, and Cloud Load Balancing with Cloud Armor.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
try:
    from google.cloud import compute_v1
    from google.cloud import secretmanager_v1
    import google.oauth2.credentials
    import googleapiclient.discovery
except ImportError:
    pass

## VPC and Firewall Rules

In [ ]:
#| export
def _compute(auth):
    return compute_v1.NetworksClient(credentials=auth.credentials)


def create_vpc(
    auth,
    name: str,
    auto_create_subnetworks: bool = False,
    **_,
) -> dict:
    """Create a custom-mode VPC network. Returns immediately if already exists."""
    client = _compute(auth)
    try:
        existing = client.get(project=auth.project, network=name)
        return {'name': name, 'self_link': existing.self_link}
    except Exception:
        pass

    network = compute_v1.Network(
        name=name,
        auto_create_subnetworks=auto_create_subnetworks,
        routing_config=compute_v1.NetworkRoutingConfig(
            routing_mode='REGIONAL',
        ),
    )
    op = client.insert(project=auth.project, network_resource=network)
    op.result(timeout=120)
    return {'name': name}


def add_subnet(
    auth,
    network_name: str,
    subnet_name: str,
    cidr: str = '10.0.0.0/24',
    region: str = None,
    private_google_access: bool = True,
    **_,
) -> dict:
    """Add a subnet to a VPC. `private_google_access=True` enables Private Google Access."""
    region = region or auth.region
    client = compute_v1.SubnetworksClient(credentials=auth.credentials)
    try:
        existing = client.get(project=auth.project, region=region, subnetwork=subnet_name)
        return {'name': subnet_name, 'cidr': existing.ip_cidr_range}
    except Exception:
        pass

    subnet = compute_v1.Subnetwork(
        name=subnet_name,
        ip_cidr_range=cidr,
        region=region,
        network=f'projects/{auth.project}/global/networks/{network_name}',
        private_ip_google_access=private_google_access,
    )
    op = client.insert(project=auth.project, region=region, subnetwork_resource=subnet)
    op.result(timeout=120)
    return {'name': subnet_name, 'cidr': cidr}


def create_firewall_rule(
    auth,
    name: str,
    network: str,
    direction: str = 'INGRESS',
    protocol: str = 'tcp',
    ports: list = None,
    source_ranges: list = None,
    target_tags: list = None,
    **_,
) -> dict:
    """Create a firewall rule. Returns immediately if the rule already exists."""
    client = compute_v1.FirewallsClient(credentials=auth.credentials)
    try:
        client.get(project=auth.project, firewall=name)
        return {'name': name}
    except Exception:
        pass

    allowed = compute_v1.Allowed(
        I_p_protocol=protocol,
        ports=ports or [],
    )
    rule = compute_v1.Firewall(
        name=name,
        network=f'projects/{auth.project}/global/networks/{network}',
        direction=direction,
        allowed=[allowed],
        source_ranges=source_ranges or (['0.0.0.0/0'] if direction == 'INGRESS' else []),
        target_tags=target_tags or [],
    )
    op = client.insert(project=auth.project, firewall_resource=rule)
    op.result(timeout=60)
    return {'name': name}

## Secret Manager

In [ ]:
#| export
def _sm(auth):
    return secretmanager_v1.SecretManagerServiceClient(credentials=auth.credentials)


def create_secret(
    auth,
    name: str,
    value: str,
    labels: dict = None,
    **_,
) -> dict:
    """Create or update a Secret Manager secret. Adds a new version with `value`."""
    client = _sm(auth)
    parent = f'projects/{auth.project}'
    secret_id = name.replace('/', '-')
    secret_name = f'{parent}/secrets/{secret_id}'

    try:
        client.get_secret(name=secret_name)
    except Exception:
        client.create_secret(
            parent=parent,
            secret_id=secret_id,
            secret=secretmanager_v1.Secret(
                replication=secretmanager_v1.Replication(
                    automatic=secretmanager_v1.Replication.Automatic()
                ),
                labels=labels or {},
            ),
        )

    version = client.add_secret_version(
        parent=secret_name,
        payload=secretmanager_v1.SecretPayload(data=value.encode()),
    )
    return {'name': secret_name, 'version': version.name}


def get_secret(auth, name: str, version: str = 'latest') -> str:
    """Retrieve the value of a Secret Manager secret version."""
    client = _sm(auth)
    secret_id = name.replace('/', '-')
    full_name = (
        f'projects/{auth.project}/secrets/{secret_id}/versions/{version}'
    )
    response = client.access_secret_version(name=full_name)
    return response.payload.data.decode()


def update_secret(auth, name: str, value: str):
    "Add a new version to an existing secret."
    create_secret(auth, name, value)


def secret_name(auth, name: str) -> str:
    "Return the full Secret Manager resource name."
    secret_id = name.replace('/', '-')
    return f'projects/{auth.project}/secrets/{secret_id}'

## Service Accounts and IAM

In [ ]:
#| export
def _iam(auth):
    return googleapiclient.discovery.build('iam', 'v1', credentials=auth.credentials)


def _crm(auth):
    return googleapiclient.discovery.build('cloudresourcemanager', 'v1',
                                           credentials=auth.credentials)


def create_service_account(
    auth,
    name: str,
    display_name: str = '',
    **_,
) -> dict:
    """Create a GCP service account. Returns existing account if already present."""
    iam = _iam(auth)
    project_name = f'projects/{auth.project}'
    email = f'{name}@{auth.project}.iam.gserviceaccount.com'

    try:
        existing = iam.projects().serviceAccounts().get(
            name=f'{project_name}/serviceAccounts/{email}'
        ).execute()
        return {'email': existing['email'], 'name': existing['name']}
    except Exception:
        pass

    body = {'accountId': name, 'serviceAccount': {'displayName': display_name or name}}
    result = iam.projects().serviceAccounts().create(
        name=project_name, body=body
    ).execute()
    return {'email': result['email'], 'name': result['name']}


def bind_iam_role(
    auth,
    member_email: str,
    role: str,
    member_type: str = 'serviceAccount',
):
    """Bind a project-level IAM role to a member (service account, user, or group)."""
    crm = _crm(auth)
    policy = crm.projects().getIamPolicy(
        resource=auth.project, body={}
    ).execute()

    member = f'{member_type}:{member_email}'
    for binding in policy.get('bindings', []):
        if binding['role'] == role:
            if member not in binding['members']:
                binding['members'].append(member)
            crm.projects().setIamPolicy(
                resource=auth.project, body={'policy': policy}
            ).execute()
            return

    policy.setdefault('bindings', []).append(
        {'role': role, 'members': [member]}
    )
    crm.projects().setIamPolicy(
        resource=auth.project, body={'policy': policy}
    ).execute()


def sa_email(auth, name: str) -> str:
    "Return the full email address for a service account in this project."
    return f'{name}@{auth.project}.iam.gserviceaccount.com'

## Private Service Connect

In [ ]:
#| export
def create_private_service_connect(
    auth,
    name: str,
    network: str,
    subnet: str,
    service_attachment: str,
    ip_address: str = None,
    **_,
) -> dict:
    """Create a Private Service Connect forwarding rule for a managed GCP service.

    `service_attachment` is the PSC service attachment URI, e.g.:
      `projects/xxx/regions/us-central1/serviceAttachments/my-service`
    """
    fr_client = compute_v1.ForwardingRulesClient(credentials=auth.credentials)
    try:
        existing = fr_client.get(project=auth.project, region=auth.region,
                                 forwarding_rule=name)
        return {'name': name, 'ip': existing.I_p_address}
    except Exception:
        pass

    body = compute_v1.ForwardingRule(
        name=name,
        network=f'projects/{auth.project}/global/networks/{network}',
        subnetwork=(
            f'projects/{auth.project}/regions/{auth.region}/subnetworks/{subnet}'
        ),
        target=service_attachment,
        load_balancing_scheme='',  # PSC uses empty string
        I_p_address=ip_address,
    )
    op = fr_client.insert(project=auth.project, region=auth.region,
                          forwarding_rule_resource=body)
    op.result(timeout=120)
    return {'name': name}

## Cloud CDN and Cloud Load Balancing

In [ ]:
#| export
def create_cdn_backend(
    auth,
    name: str,
    bucket_name: str,
    cdn_policy: dict = None,
    **_,
) -> dict:
    """Create a Cloud CDN backend bucket for serving static content."""
    client = compute_v1.BackendBucketsClient(credentials=auth.credentials)
    try:
        existing = client.get(project=auth.project, backend_bucket=name)
        return {'name': name, 'self_link': existing.self_link}
    except Exception:
        pass

    backend = compute_v1.BackendBucket(
        name=name,
        bucket_name=bucket_name,
        enable_cdn=True,
        cdn_policy=compute_v1.BackendBucketCdnPolicy(
            **(cdn_policy or {})
        ),
    )
    op = client.insert(project=auth.project, backend_bucket_resource=backend)
    op.result(timeout=120)
    return {'name': name}


def create_https_lb(
    auth,
    name: str,
    backend_service: str,
    armor_policy: str = None,
    **_,
) -> dict:
    """Create a global HTTPS load balancer with optional Cloud Armor security policy.

    Creates: URL map → target HTTPS proxy → global forwarding rule.
    Pass `armor_policy` as the full resource URL of a Cloud Armor security policy.
    """
    compute = googleapiclient.discovery.build(
        'compute', 'v1', credentials=auth.credentials
    )
    project = auth.project

    # URL map
    url_map_name = f'{name}-url-map'
    try:
        compute.urlMaps().get(project=project, urlMap=url_map_name).execute()
    except Exception:
        compute.urlMaps().insert(project=project, body={
            'name': url_map_name,
            'defaultService': backend_service,
        }).execute()

    # HTTPS proxy (requires SSL cert — omitted here for brevity; use managed cert)
    proxy_name = f'{name}-https-proxy'
    try:
        compute.targetHttpsProxies().get(project=project,
                                         targetHttpsProxy=proxy_name).execute()
    except Exception:
        compute.targetHttpsProxies().insert(project=project, body={
            'name': proxy_name,
            'urlMap': f'global/urlMaps/{url_map_name}',
            'sslCertificates': [],  # attach managed cert separately
        }).execute()

    # Global forwarding rule
    fr_name = f'{name}-fr'
    try:
        fr = compute.globalForwardingRules().get(
            project=project, forwardingRule=fr_name
        ).execute()
        return {'name': fr_name, 'ip': fr.get('IPAddress')}
    except Exception:
        pass

    body = {
        'name': fr_name,
        'target': f'global/targetHttpsProxies/{proxy_name}',
        'portRange': '443',
        'IPProtocol': 'TCP',
        'loadBalancingScheme': 'EXTERNAL_MANAGED',
    }
    if armor_policy:
        body['securityPolicy'] = armor_policy
    op = compute.globalForwardingRules().insert(project=project, body=body).execute()
    return {'name': fr_name, 'operation': op.get('name')}